# 🔬 SSD300 — Esophageal Cancer Detection
### 150 Epochs | Loss Curves | Confusion Matrix | Metrics Table | best.pt

In [ ]:
import os
os.system('pip install pycocotools -q')
print('✅ Done')

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict
import torch, torchvision
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')
print(f'PyTorch:{torch.__version__}  GPU:{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ═══ CONFIG ← only change COCO_ROOT if needed ═════════════
COCO_ROOT  = '/kaggle/input/datasets/rugvedkulkarni2509/esophageal-balanced-coco/coco_balanced'
ANNOTS_DIR = COCO_ROOT + '/annotations'
OUTPUT_DIR = '/kaggle/working/ssd_output'
EPOCHS=150; BATCH_SIZE=8; LR=0.01
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(OUTPUT_DIR, exist_ok=True)
# Auto-detect classes
with open(f'{ANNOTS_DIR}/instances_train.json') as f: _d=json.load(f)
CLASS_NAMES=[c['name'] for c in sorted(_d['categories'],key=lambda x:x['id'])]
NUM_CLASSES=len(CLASS_NAMES)+1
print(f'✅ Config OK  classes:{CLASS_NAMES}  nc:{NUM_CLASSES}  device:{DEVICE}')
for sp in ['train','val','test']:
    with open(f'{ANNOTS_DIR}/instances_{sp}.json') as f: dd=json.load(f)
    print(f'  ✅ {sp:5s} → {len(dd["images"])} images | {len(dd["annotations"])} annotations')

In [ ]:
# ═══ Dataset ══════════════════════════════════════════════
class RobustCocoDS(Dataset):
    def __init__(self,img_dir,ann_file,size=300,tfm=None):
        self.img_dir=Path(img_dir); self.size=size; self.tfm=tfm
        with open(ann_file) as f: coco=json.load(f)
        cats=sorted(coco['categories'],key=lambda x:x['id'])
        self.cat_map={c['id']:i+1 for i,c in enumerate(cats)}
        abi=defaultdict(list)
        for a in coco['annotations']: abi[a['image_id']].append(a)
        self.samples=[]
        for img in coco['images']:
            p=self.img_dir/img['file_name']
            if not p.exists(): continue
            va=[a for a in abi[img['id']] if a['bbox'][2]>=10 and a['bbox'][3]>=10]
            if va: self.samples.append((img,va))
        print(f'  ✅ {len(self.samples)} valid images loaded')
    def __len__(self): return len(self.samples)
    def __getitem__(self,idx):
        info,anns=self.samples[idx]
        img=Image.open(self.img_dir/info['file_name']).convert('RGB')
        W,H=img.size
        boxes,labels=[],[]
        for a in anns:
            x,y,w,h=a['bbox']
            x1=max(0.,x); y1=max(0.,y)
            x2=min(float(W),x1+w); y2=min(float(H),y1+h)
            if x2-x1>=5 and y2-y1>=5:
                boxes.append([x1,y1,x2,y2]); labels.append(self.cat_map[a['category_id']])
        if not boxes: boxes=[[0.,0.,float(W),float(H)]]; labels=[1]
        target={'boxes':torch.tensor(boxes,dtype=torch.float32),'labels':torch.tensor(labels,dtype=torch.int64),'image_id':torch.tensor([idx])}
        if self.tfm: img=self.tfm(img)
        return img,target

def collate_fn(b): return tuple(zip(*b))
tfm=T.Compose([T.Resize((300,300)),T.ToTensor()])
train_ds=RobustCocoDS(f'{COCO_ROOT}/train',f'{ANNOTS_DIR}/instances_train.json',300,tfm)
val_ds  =RobustCocoDS(f'{COCO_ROOT}/val',  f'{ANNOTS_DIR}/instances_val.json',  300,tfm)
test_ds =RobustCocoDS(f'{COCO_ROOT}/test', f'{ANNOTS_DIR}/instances_test.json', 300,tfm)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True, collate_fn=collate_fn,num_workers=2)
val_loader  =DataLoader(val_ds,  batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2)
test_loader =DataLoader(test_ds, batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=2)
print(f'✅ Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}')

In [ ]:
# ═══ Build SSD ════════════════════════════════════════════
model=ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)
in_channels=[512,1024,512,256,256,256]
num_anchors=model.anchor_generator.num_anchors_per_location()
model.head.classification_head=SSDClassificationHead(in_channels,num_anchors,NUM_CLASSES)
model=model.to(DEVICE)
optimizer=torch.optim.SGD(model.parameters(),lr=LR,momentum=0.9,weight_decay=5e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
print(f'✅ SSD300 built  params:{sum(p.numel() for p in model.parameters()):,}  nc:{NUM_CLASSES}')

In [ ]:
# ═══ Train ════════════════════════════════════════════════
train_losses,val_losses=[],[]
best_val=float('inf'); best_pt=os.path.join(OUTPUT_DIR,'best.pt')

def run_epoch(model,loader,optimizer=None):
    model.train(); total,count=0,0
    for imgs,tgts in loader:
        try:
            imgs=[i.to(DEVICE) for i in imgs]
            tgts=[{k:v.to(DEVICE) for k,v in t.items()} for t in tgts]
            bad=any(t['boxes'].shape[0]==0 or t['labels'].max()>=NUM_CLASSES or t['labels'].min()<1 for t in tgts)
            if bad: continue
            if optimizer:
                ld=model(imgs,tgts); loss=sum(ld.values())
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),10.)
                optimizer.step()
            else:
                with torch.no_grad(): ld=model(imgs,tgts); loss=sum(ld.values())
            total+=loss.item(); count+=1
        except Exception: continue
    return total/max(count,1)

print('Starting SSD Training...'); print('='*60)
for ep in range(1,EPOCHS+1):
    tl=run_epoch(model,train_loader,optimizer)
    vl=run_epoch(model,val_loader)
    scheduler.step(); train_losses.append(tl); val_losses.append(vl)
    if vl<best_val: best_val=vl; torch.save(model.state_dict(),best_pt)
    if ep%10==0 or ep==1: print(f'Epoch {ep:>3}/{EPOCHS}  train={tl:.4f}  val={vl:.4f}  best={best_val:.4f}')
print(f'\n✅ Done! best.pt → {best_pt}')

In [ ]:
# ═══ Loss Curves ══════════════════════════════════════════
fig,ax=plt.subplots(figsize=(12,5))
ax.plot(range(1,len(train_losses)+1),train_losses,label='Train',color='#2196F3',linewidth=1.8)
ax.plot(range(1,len(val_losses)+1),  val_losses,  label='Val',  color='#F44336',linewidth=1.8)
ax.set_title('SSD300 — Esophageal Cancer — Loss Curves',fontweight='bold',fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'ssd_loss_curves.png'),dpi=150,bbox_inches='tight'); plt.show()
pd.DataFrame({'epoch':range(1,len(train_losses)+1),'train':train_losses,'val':val_losses}).to_csv(os.path.join(OUTPUT_DIR,'ssd_losses.csv'),index=False)
print('✅ Loss curves saved')

In [ ]:
# ═══ Evaluate + CM ════════════════════════════════════════
model.load_state_dict(torch.load(best_pt,map_location=DEVICE)); model.eval()
nc=len(CLASS_NAMES); cm_array=np.zeros((nc,nc),dtype=int)
def box_iou(a,b):
    xA=max(a[0],b[0]);yA=max(a[1],b[1]);xB=min(a[2],b[2]);yB=min(a[3],b[3])
    i=max(0,xB-xA)*max(0,yB-yA); u=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-i
    return i/u if u>0 else 0
with torch.no_grad():
    for imgs,tgts in test_loader:
        imgs=[i.to(DEVICE) for i in imgs]; preds=model(imgs)
        for pred,tgt in zip(preds,tgts):
            gb=tgt['boxes'].numpy(); gl=tgt['labels'].numpy()
            keep=pred['scores'].cpu().numpy()>0.5
            pb=pred['boxes'].cpu().numpy()[keep]; pl=pred['labels'].cpu().numpy()[keep]
            mg=set()
            for pbox,plbl in zip(pb,pl):
                bi,bv=-1,0
                for gi,(gbox,glbl) in enumerate(zip(gb,gl)):
                    if gi in mg: continue
                    v=box_iou(pbox,gbox)
                    if v>bv: bv=v; bi=gi
                if bv>=0.5 and bi>=0:
                    gc=gl[bi]-1; pc=plbl-1
                    if 0<=gc<nc and 0<=pc<nc: cm_array[gc,pc]+=1
                    mg.add(bi)
print('✅ Eval done'); print(cm_array)
cm_n=cm_array.astype(float); rs=cm_n.sum(axis=1,keepdims=True); rs[rs==0]=1; cm_n/=rs
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle('SSD300 — Confusion Matrix — Esophageal Cancer',fontsize=13,fontweight='bold')
for ax,d,t,f in zip(axes,[cm_array,cm_n],['Raw Counts','Normalised'],['d','.2f']):
    sns.heatmap(d,annot=True,fmt=f,cmap='Blues',xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,
                ax=ax,linewidths=0.8,annot_kws={'size':13,'weight':'bold'})
    ax.set_title(t,fontweight='bold'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'ssd_confusion_matrix.png'),dpi=150,bbox_inches='tight'); plt.show()
print('✅ Confusion matrix saved')

In [ ]:
# ═══ Metrics Table ════════════════════════════════════════
def compute_metrics(cm,names):
    rows=[]; total=cm.sum()
    for i,cls in enumerate(names[:cm.shape[0]]):
        TP=cm[i,i]; FP=cm[:,i].sum()-TP; FN=cm[i,:].sum()-TP; TN=total-TP-FP-FN
        P=TP/(TP+FP) if TP+FP>0 else 0; R=TP/(TP+FN) if TP+FN>0 else 0
        F1=2*P*R/(P+R) if P+R>0 else 0; Acc=(TP+TN)/total if total>0 else 0
        rows.append({'Class':cls,'TP':int(TP),'FP':int(FP),'FN':int(FN),'TN':int(TN),
                     'Precision':round(P,4),'Recall':round(R,4),'F1-Score':round(F1,4),'Accuracy':round(Acc,4)})
    rows.append({'Class':'MACRO AVG','TP':'','FP':'','FN':'','TN':'',
                 'Precision':round(np.mean([r['Precision'] for r in rows]),4),
                 'Recall':round(np.mean([r['Recall'] for r in rows]),4),
                 'F1-Score':round(np.mean([r['F1-Score'] for r in rows]),4),
                 'Accuracy':round(np.mean([r['Accuracy'] for r in rows]),4)})
    return pd.DataFrame(rows)
mdf=compute_metrics(cm_array.astype(float),CLASS_NAMES)
print('='*65); print('  SSD METRICS'); print('='*65); print(mdf.to_string(index=False)); print('='*65)
fig,ax=plt.subplots(figsize=(15,max(3,len(mdf)*0.65+1.2))); ax.axis('off')
tbl=ax.table(cellText=mdf.values.tolist(),colLabels=list(mdf.columns),cellLoc='center',loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2,1.8)
for j in range(len(mdf.columns)): tbl[0,j].set_facecolor('#1565C0'); tbl[0,j].set_text_props(color='white',fontweight='bold')
for i in range(1,len(mdf)):
    for j in range(len(mdf.columns)): tbl[i,j].set_facecolor('#F5F5F5' if i%2==0 else 'white')
last=len(mdf)
for j in range(len(mdf.columns)): tbl[last,j].set_facecolor('#BBDEFB'); tbl[last,j].set_text_props(fontweight='bold')
ax.set_title('SSD300 — Precision/Recall/F1/Accuracy',fontsize=13,fontweight='bold',pad=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'ssd_metrics_table.png'),dpi=150,bbox_inches='tight'); plt.show()
mdf.to_csv(os.path.join(OUTPUT_DIR,'ssd_metrics.csv'),index=False); print('✅ Metrics saved')

In [ ]:
print('='*60); print('  SSD OUTPUT FILES'); print('='*60)
for n,p in [('best.pt',best_pt),('Loss Curves',os.path.join(OUTPUT_DIR,'ssd_loss_curves.png')),
             ('Confusion Matrix',os.path.join(OUTPUT_DIR,'ssd_confusion_matrix.png')),
             ('Metrics Table',os.path.join(OUTPUT_DIR,'ssd_metrics_table.png')),
             ('Metrics CSV',os.path.join(OUTPUT_DIR,'ssd_metrics.csv'))]:
    print(f'  {"✅" if os.path.exists(p) else "❌"}  {n:<22} → {p}')
print('\n🎉 SSD Training Complete!')